# Results analysis

Optional exploratory notebook. It does not run any experiments itself -- it only loads the JSON tables written by the scripts in `experiments/` and plots them. Run at least one of the following first, from the repository root:

```bash
python experiments/ideal_benchmark.py
python experiments/scalability.py
python experiments/noise_sweeps.py
```

(add `--quick` to any of them for a fast, non-representative run). Each cell below is independent and will simply report a missing file if you haven't run the corresponding script.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

RESULTS = Path("../results")
COLORS = {"stokes": "#4C72B0", "mle": "#55A868", "nn": "#C44E52"}


def load(name):
    path = RESULTS / "tables" / name
    if not path.exists():
        print(f"Missing {path} -- run the corresponding experiment script first.")
        return None
    with open(path) as f:
        return json.load(f)

## Ideal two-qubit benchmark (report Fig. 6)

In [ ]:
data = load("ideal_benchmark.json")
if data:
    methods = ["stokes", "mle", "nn"]
    means = [data[m]["mean_fidelity"] for m in methods]
    stds = [data[m]["std_fidelity"] for m in methods]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(methods, means, yerr=stds, capsize=4, color=[COLORS[m] for m in methods])
    ax.set_ylabel("Mean fidelity")
    ax.set_ylim(0, 1.05)
    plt.show()

## Scalability with qubit number (report Fig. 8)

In [ ]:
records = []
for n in (2, 3, 4, 5):
    d = load(f"scalability_{n}qubit.json")
    if d:
        records.append(d)

if records:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    ns = [r["n_qubits"] for r in records]
    for method, color in COLORS.items():
        ax1.errorbar(
            ns,
            [r[method]["mean_fidelity"] for r in records],
            yerr=[r[method]["std_fidelity"] for r in records],
            label=method.upper(), color=color, marker="o",
        )
        ax2.errorbar(
            ns,
            [r[method]["mean_time"] for r in records],
            yerr=[r[method]["std_time"] for r in records],
            label=method.upper(), color=color, marker="o",
        )
    ax1.set_xlabel("Number of qubits"); ax1.set_ylabel("Mean fidelity"); ax1.legend()
    ax2.set_xlabel("Number of qubits"); ax2.set_ylabel("Mean inference time (s)"); ax2.set_yscale("log"); ax2.legend()
    plt.show()

## Noise sweeps (report Fig. 7)

In [ ]:
channels = {"sigma": "Additive Gaussian std", "shots": "No. of shots", "pepper": "Pepper fraction"}
loaded = {c: load(f"noise_sweep_{c}.json") for c in channels}
loaded = {c: d for c, d in loaded.items() if d}

if loaded:
    fig, axes = plt.subplots(1, len(loaded), figsize=(5 * len(loaded), 4), squeeze=False)
    for ax, (channel, data) in zip(axes[0], loaded.items()):
        x = [p["value"] for p in data]
        for method, color in COLORS.items():
            ax.errorbar(
                x,
                [p[f"{method}_mean_fidelity"] for p in data],
                yerr=[p[f"{method}_std_fidelity"] for p in data],
                label=method.upper(), color=color, marker="o",
            )
        if channel == "shots":
            ax.set_xscale("log")
        ax.set_xlabel(channels[channel]); ax.set_ylabel("Mean fidelity"); ax.legend()
    plt.show()